In [1]:
import requests
import dask
import dask.dataframe
#import dask.distributed
import numpy
import pandas
import json
import logging
from typing import Tuple, Dict

import shapely.geometry

import dataservice

/opt/miniconda3/envs/dask/lib/python3.9/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(


In [3]:
#client = dask.distributed.Client('127.0.0.1:8786')

In [4]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

### Globals

In [5]:
dataserviceendpoint = 'http://pairs-interactive01.pok.ibm.com:9082/pairsdataservice'

GLOBAL_AREA = shapely.geometry.box(-180, -90, 180, 90)
DELTA_PIXEL_QUERY = 11
DELTA_PIXEL_CELL = 5
DELTA_CELL_QUERY = 6
PERCENTILES = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

In [6]:
dataframe_meta = pandas.DataFrame.from_dict(
    {
        'timestamp' : pandas.Series(dtype=numpy.int64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'first' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'sum' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'count' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'mean' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'std' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'min' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '1%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '5%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '10%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '25%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '50%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '75%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '90%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '95%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        '99%' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64)),
        'max' : pandas.Series(dtype=numpy.float64, index=pandas.Index([], name='key', dtype=numpy.int64))        
    }
)

In [7]:
@dask.delayed
def to_dask_frame(layer_id: str, level: int, key: int, timestamp: int, dimensions: Dict={}, overview=False, xy=False):
    return dataservice.query_key_to_frame(layer_id, level, key, timestamp, dimensions, overview, xy)

def to_overview_frame(layer_id, level, timestamp, dimensions: Dict={}):
    _, query_keys = dataservice.quadtree(GLOBAL_AREA, level-DELTA_PIXEL_QUERY)
    assert sorted(query_keys)==query_keys

    dask_frame = dask.dataframe.concat(
        [
            dask.dataframe.from_delayed(
                to_dask_frame(layer_id, level, key, timestamp, overview=True),
                meta=dataframe_meta,
                verify_meta=False
            )
            for key in query_keys
        ], axis=0
    )
    
    return dask_frame

In [8]:
layer_id = '49257' # gCOD Temperature
layer_id = '49459' # ERA5 precipitation
starttime = '2020-01-01T00:00:00'
endtime = '2020-01-02T00:00:00'
level = 14 # gCOD
level = 12 # ERA5

In [9]:
timestamps = dataservice.get_global_timestamps(layer_id, starttime, endtime)

In [10]:
data_frame = dask.dataframe.concat(
    [
        to_overview_frame(layer_id, level, t)
        for t in timestamps
    ], axis=0
)

Write to disk:

In [11]:
data_frame.to_parquet(
    f'parquet/layer{layer_id}_level{level-5}', engine='pyarrow', write_index=True,
)

Write to COS:

In [15]:
with open('cos_credentials.json') as fp:
    storage_options = json.load(fp)

In [14]:
data_frame.to_parquet(
    f's3://geolab-extended-overviews/layer{layer_id}_level{level-5}', engine='pyarrow', write_index=True,
    storage_options=storage_options
)